## 1. GPU check

In [8]:
import subprocess
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if out.returncode == 0 and out.stdout.strip():
    print(out.stdout.strip())
    print("\nGPU found. Good.")
else:
    print("!! NO GPU DETECTED !!")
    print("Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    print("You can continue on CPU, but expect many hours. If you must, set")
    print("MODEL = 'llama3.2:3b' in cell 5.")

GPU 0: Tesla T4 (UUID: GPU-0dd994fe-f1bd-ebff-2b74-92ece777c48f)

GPU found. Good.


## 2. Python dependencies


In [9]:
!pip install -q "autogen-agentchat==0.7.5" "autogen-ext[openai]==0.7.5" "datasets>=4.0" pandas scipy matplotlib
print("\ndependencies installed")


dependencies installed


## 3. Ollama — install, serve, pull the model

In [10]:
import os, subprocess, time, urllib.request, shutil

WORK = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(WORK)

MODEL = "llama3.1:8b"   # the model the dissertation used. "llama3.2:3b" is ~3x faster.

BIN_DIRS = ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]
os.environ["PATH"] = ":".join(BIN_DIRS) + ":" + os.environ.get("PATH", "")


def ollama_bin():
    p = shutil.which("ollama")
    if p:
        return p
    for d in BIN_DIRS:
        c = os.path.join(d, "ollama")
        if os.path.exists(c):
            return c
    return None


# Ollama releases are zstd-compressed and the Colab image ships no zstd binary,
# so the installer aborts without it. Install it first.
if shutil.which("zstd") is None:
    print("installing zstd (Ollama needs it to unpack)...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, check=True)
print("zstd:", shutil.which("zstd"))

if ollama_bin() is None:
    # Attempt 1: the official installer. It can exit non-zero in Colab (no systemd,
    # no user/group management) even when it did install the binary, so we check
    # for the binary afterwards rather than trusting the exit code.
    print("\nattempt 1: official install script")
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True, text=True)
    print("exit code:", r.returncode)
    if r.stdout.strip():
        print("--- stdout ---\n" + r.stdout[-2500:])
    if r.stderr.strip():
        print("--- stderr ---\n" + r.stderr[-2500:])

if ollama_bin() is None:
    # Attempt 2: unpack the release archive by hand (note .tar.zst -- the old .tgz
    # assets are gone, which is why a .tgz URL 404s).
    print("\nattempt 2: direct archive install")
    url = "https://ollama.com/download/ollama-linux-amd64.tar.zst"
    subprocess.run(f"curl -fL --retry 3 {url} | tar -x --zstd -C /usr",
                   shell=True, check=True)

if ollama_bin() is None:
    raise RuntimeError("Could not install Ollama. Read the output above for the reason.")

OLLAMA = ollama_bin()
print("\nollama binary:", OLLAMA)
subprocess.run([OLLAMA, "--version"])

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"        # don't unload between calls
os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"   # prompts here exceed the 2048 default
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"


def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False


if not ollama_up():
    log = open(os.path.join(WORK, "ollama.log"), "w")
    subprocess.Popen([OLLAMA, "serve"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(90):
        if ollama_up():
            break
        time.sleep(1)

if not ollama_up():
    raise RuntimeError("Ollama did not start. Check /content/ollama.log")
print("ollama server is up")

print(f"pulling {MODEL} (a few minutes the first time)...")
subprocess.run([OLLAMA, "pull", MODEL], check=True)
subprocess.run([OLLAMA, "list"])

installing zstd (Ollama needs it to unpack)...
zstd: /usr/bin/zstd

attempt 1: official install script
exit code: 0
--- stdout ---

--- stderr ---
###############################################                      73.5%
#####################################################                     74.3%
######################################################                    75.3%
######################################################                    76.2%
#######################################################                   76.8%
########################################################                  77.8%
########################################################                  78.7%
#########################################################                 79.6%
##########################################################                80.6%
##########################################################                81.7%
##########################################################

CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

## 4. Get the code


In [11]:
import os, subprocess, glob, zipfile

os.chdir(WORK)
PROJECT = None

repo_dir = os.path.join(WORK, "LLM-Arbitration")
if not os.path.isdir(repo_dir):
    try:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/y6s19/LLM-Arbitration.git"], check=True)
    except Exception as e:
        print("git clone failed:", e)

if os.path.isdir(os.path.join(repo_dir, "LLM_Arbitration", "code")):
    PROJECT = os.path.join(repo_dir, "LLM_Arbitration")

if PROJECT is None:                       # fall back to the uploaded zip
    zips = glob.glob(os.path.join(WORK, "*.zip"))
    if not zips:
        raise RuntimeError("No repo and no zip. Upload final_project_v2.zip to /content.")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(os.path.join(WORK, "from_zip"))
    hits = glob.glob(os.path.join(WORK, "from_zip", "**", "pipeline.py"), recursive=True)
    PROJECT = os.path.dirname(os.path.dirname(hits[0]))
    print("using uploaded zip:", zips[0])

CODE = os.path.join(PROJECT, "code")
ANALYSIS = os.path.join(PROJECT, "analysis", "scripts")
os.makedirs(os.path.join(CODE, "logs"), exist_ok=True)

print("project :", PROJECT)
print("code    :", CODE)
print("code files:", len([f for f in os.listdir(CODE) if f.endswith(".py")]))

project : /content/LLM-Arbitration/LLM_Arbitration
code    : /content/LLM-Arbitration/LLM_Arbitration/code
code files: 15


## 5. Settings + the driver script


In [12]:
N_TRIALS    = 5     # scenarios / problems / pairs per experiment (the dissertation used 5)
SLEEP_SCALE = 0.0   # 0.0 = no rate-limit pauses (local model), 1.0 = original Groq timings

DRIVER_SRC = r'''
# _colab_driver.py -- single entry point for all three experiments.
# Adapts the original scripts (written for Groq / a Windows Ollama box)
# to run unchanged against a local Ollama server inside Colab.
import argparse, asyncio, os, sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

ap = argparse.ArgumentParser()
ap.add_argument("--experiment", required=True, choices=["casino", "gsm8k", "multiwoz"])
ap.add_argument("--model", default="llama3.1:8b")
ap.add_argument("--n", type=int, default=5)
ap.add_argument("--sleep-scale", type=float, default=0.0)
args = ap.parse_args()

# ---------------------------------------------------------------- 1. model
# Every pipeline already calls experiment_utils.make_ollama_client(); we only
# override the default model name so it can be swapped from the notebook.
import experiment_utils as eu
_orig_make = eu.make_ollama_client
eu.make_ollama_client = lambda model=args.model: _orig_make(model)

# ---------------------------------------------------------------- 2. pacing
# The pipelines sleep 2-10s between calls to stay under Groq's free-tier rate
# limit. A local Ollama server has no such limit, so those pauses are pure
# wall-clock waste. --sleep-scale 1.0 restores the original timings exactly.
import asyncio as _aio


class _SleepShim:
    def __init__(self, scale):
        self.scale = scale

    def __getattr__(self, name):
        return getattr(_aio, name)

    async def sleep(self, delay, *a, **kw):
        return await _aio.sleep(max(delay * self.scale, 0))


shim = _SleepShim(args.sleep_scale)

import pipeline
pipeline.asyncio = shim

# ---------------------------------------------------------------- 3. dispatch
if args.experiment == "casino":
    import run_experiment as mod
    mod.asyncio = shim
    mod.SCENARIO_INDICES = list(range(args.n))
    label = "CaSiNo (negotiation)"
elif args.experiment == "gsm8k":
    import pipeline_gsm8k
    pipeline_gsm8k.asyncio = shim
    import run_experiment_gsm8k as mod
    mod.asyncio = shim
    mod.PROBLEM_INDICES = list(range(args.n))
    label = "GSM8K (maths)"
else:
    import pipeline_multiwoz
    pipeline_multiwoz.asyncio = shim
    import run_experiment_multiwoz as mod
    mod.asyncio = shim
    mod.PAIR_INDICES = list(range(args.n))
    label = "MultiWOZ (trip planning)"

print("=" * 70)
print(f"{label}   model={args.model}   trials={args.n}   sleep_scale={args.sleep_scale}")
print("=" * 70, flush=True)

asyncio.run(mod.main())
'''

driver_path = os.path.join(CODE, "_colab_driver.py")
with open(driver_path, "w") as f:
    f.write(DRIVER_SRC)
print("wrote", driver_path)
print(f"MODEL={MODEL}  N_TRIALS={N_TRIALS}  SLEEP_SCALE={SLEEP_SCALE}")


import sys, time


def run(experiment, n=None):
    # Run one experiment as a subprocess, streaming its output live.
    n = N_TRIALS if n is None else n
    cmd = [sys.executable, "_colab_driver.py", "--experiment", experiment,
           "--model", MODEL, "--n", str(n), "--sleep-scale", str(SLEEP_SCALE)]
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n--- {experiment}: exit {p.returncode}, {(time.time()-t0)/60:.1f} min ---")
    return p.returncode

wrote /content/LLM-Arbitration/LLM_Arbitration/code/_colab_driver.py
MODEL=llama3.1:8b  N_TRIALS=5  SLEEP_SCALE=0.0


## 6. Smoke test


In [13]:
smoke = r'''
import asyncio, sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import experiment_utils as eu
_m = eu.make_ollama_client
eu.make_ollama_client = lambda model=sys.argv[1]: _m(model)
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
import pipeline
from experiment_utils import transcript_to_text

async def main():
    ca, cb = eu.make_ollama_client(), eu.make_ollama_client()
    a = AssistantAgent(name="Camper_A", model_client=ca,
                       system_message="You are Camper A. Keep replies to one sentence.")
    b = AssistantAgent(name="Camper_B", model_client=cb,
                       system_message="You are Camper B. Keep replies to one sentence.")
    team = RoundRobinGroupChat([a, b],
                               termination_condition=MaxMessageTermination(max_messages=4))
    r = await team.run(task="Split 3 food packages between you. Be brief.")
    print(transcript_to_text(r.messages))
    lab, _ = await pipeline.classify_turn(a, "", "Camper_A", "Fine, I agree, deal.", False)
    print("\nclassifier label ->", lab)
    await ca.close(); await cb.close()

asyncio.run(main())
'''
with open(os.path.join(CODE, "_smoke.py"), "w") as f:
    f.write(smoke)

rc = subprocess.run([sys.executable, "_smoke.py", MODEL], cwd=CODE)
print("\nSMOKE TEST", "PASSED" if rc.returncode == 0 else "FAILED")


SMOKE TEST PASSED


## 7. Experiment 1 — CaSiNo (negotiation)


In [14]:
run("casino")

CaSiNo (negotiation)   model=llama3.1:8b   trials=5   sleep_scale=0.0

TRIAL: scenario 0


Generating train split: 100%|██████████| 1030/1030 [00:00<00:00, 14684.89 examples/s]
=== [Scenario 0] Running negotiation (once) ===

Camper_A: I agree to prioritize the Firewood packages, taking two of the three. With our large group and absence of other light sources, we need all the firewood we can get.

Camper_B: I'll give you two Firewood packages then. I'll take one, which will help with my flea issue. With two packages going to the group, can we discuss splitting the Water packages? I think one Water package would still be sufficient for me.

Camper_A: Thank you for the two Firewood packages. As for Water, I think one Water package will indeed be sufficient. I'd like to keep two Food packages as well, one for each of my campsite's meals.

Camper_B: Since I don't need as much food, I'm willing to split the Food packages with one going to the group. Can I have one of the Food packages, and 

0

## 8. Experiment 2 — GSM8K

In [15]:
run("gsm8k")

GSM8K (maths)   model=llama3.1:8b   trials=5   sleep_scale=0.0

GSM8K TRIAL: problem 0


Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 161489.77 examples/s]

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 155633.02 examples/s]
=== [GSM8K problem 0] Question ===
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
(Ground truth: 18)

Solver_A: **Solver_A's working:**

1. Janet's ducks lay 16 eggs per day, and she eats 3 for breakfast and bakes 4.
2. Total used eggs = 3 + 4 = 7 eggs.
3. Remaining eggs to sell = Total eggs - Used eggs = 16 - 7 = 9 eggs.
4. Janet sells 9 eggs at $2 each, so her daily earnings are 9 x $2 = $18.

**Solver_A's answer:** FINAL ANSWER: 18

**Would you like to challenge or agree?

Solver_B: **Solver_B's

0

## 9. Experiment 3 — MultiWOZ

In [16]:
run("multiwoz")

MultiWOZ (trip planning)   model=llama3.1:8b   trials=5   sleep_scale=0.0

MULTIWOZ TRIAL: pair 0


Generating train split: 100%|██████████| 8437/8437 [00:01<00:00, 7649.41 examples/s]

Generating validation split: 100%|██████████| 1000/1000 [00:00<00:00, 8597.37 examples/s]

Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 9267.42 examples/s]
=== [MultiWOZ pair 0] ===
Traveler_A's real preference: i need a place to dine in the center thats expensive
Traveler_B's real preference: Yeah, I need a restaurant in the west and with expensive pricing.

Traveler_A: I'm thinking of a fancy restaurant in the center. How about we go to Le Comptoir du Relais, it's pricey and located near the heart of the city. Would you be up for trying it out?

Traveler_B: I think you might be misunderstanding. I'm looking for a restaurant in the West, not the center. But I'm up for trying Le Comptoir du Relais anyway, even though it's not in the West.

Traveler_A: You're right, Le Comptoir du Rela

0

## 10. Your fresh results


In [17]:
import glob, pandas as pd

LOGS = os.path.join(CODE, "logs")


def newest(pattern):
    hits = sorted(glob.glob(os.path.join(LOGS, pattern)))
    return hits[-1] if hits else None


for name, pattern in [("CaSiNo", "experiment_summary_*.csv"),
                      ("GSM8K", "gsm8k_summary_*.csv"),
                      ("MultiWOZ", "multiwoz_summary_*.csv")]:
    path = newest(pattern)
    if not path:
        print(f"{name}: no CSV found (did that experiment run?)\n")
        continue
    df = pd.read_csv(path)
    print("=" * 60)
    print(f"{name}   n={len(df)}   {os.path.basename(path)}")
    print("=" * 60)
    for m in ["voting", "structured", "judge"]:
        if m in df:
            k = (df[m] == "APPROVED").sum()
            print(f"  {m:<11} {k}/{len(df)} approved ({100*k/len(df):.0f}%)")
    if "task_correct" in df:
        k = (df["task_correct"] == True).sum()
        print(f"  task accuracy {k}/{len(df)}")
    lab = [c for c in ["agree", "disagree", "stall", "escalate", "unclear"] if c in df]
    if lab:
        print("  turn labels:", df[lab].sum().to_dict())
    print()

print("Per-trial transcripts and reasoning are in:", LOGS)

CaSiNo   n=5   experiment_summary_20260919_005650.csv
  voting      5/5 approved (100%)
  structured  3/5 approved (60%)
  judge       0/5 approved (0%)
  turn labels: {'agree': 27, 'disagree': 10, 'stall': 1, 'escalate': 0, 'unclear': 7}

GSM8K   n=5   gsm8k_summary_20260919_010343.csv
  voting      5/5 approved (100%)
  structured  5/5 approved (100%)
  judge       4/5 approved (80%)
  task accuracy 4/5
  turn labels: {'agree': 17, 'disagree': 3, 'stall': 4, 'escalate': 0, 'unclear': 1}

MultiWOZ   n=5   multiwoz_summary_20260919_011030.csv
  voting      5/5 approved (100%)
  structured  3/5 approved (60%)
  judge       0/5 approved (0%)
  turn labels: {'agree': 19, 'disagree': 6, 'stall': 3, 'escalate': 0, 'unclear': 7}

Per-trial transcripts and reasoning are in: /content/LLM-Arbitration/LLM_Arbitration/code/logs


## 11. Analysis + figures


In [18]:
for script in ["analyse.py", "accuracy.py", "generate_figures.py"]:
    path = os.path.join(ANALYSIS, script)
    if not os.path.exists(path):
        print(f"(skipping {script} — not present; it lives in the git repo, not the zip)")
        continue
    print("\n" + "#" * 70)
    print("#", script)
    print("#" * 70)
    subprocess.run([sys.executable, script], cwd=ANALYSIS)

figs = glob.glob(os.path.join(ANALYSIS, "figures", "*.svg"))
print(f"\n{len(figs)} figures written to {os.path.join(ANALYSIS, 'figures')}")
for f in sorted(figs):
    print("  ", os.path.basename(f))


######################################################################
# analyse.py
######################################################################

######################################################################
# accuracy.py
######################################################################

######################################################################
# generate_figures.py
######################################################################

6 figures written to /content/LLM-Arbitration/LLM_Arbitration/analysis/scripts/figures
   figure_3_1_paired_design.svg
   figure_4_1_architecture.svg
   figure_4_2_classifier_iterations.svg
   figure_5_1_label_distribution.svg
   figure_5_2_verdict_accuracy.svg
   figure_5_3_verdict_matrix.svg


## 12. Download everything

In [19]:
import shutil

bundle = os.path.join(WORK, "arbitration_results")
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle, exist_ok=True)

shutil.copytree(LOGS, os.path.join(bundle, "new_run_logs"))
fig_dir = os.path.join(ANALYSIS, "figures")
if os.path.isdir(fig_dir):
    shutil.copytree(fig_dir, os.path.join(bundle, "figures"))

archive = shutil.make_archive(bundle, "zip", bundle)
print("bundled:", archive, f"({os.path.getsize(archive)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("(download it from the file browser on the left)", e)

bundled: /content/arbitration_results.zip (0.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>